# Decomposing Supply- and Demand-Driven Inflation

Replication of **Shapiro (2024), FRBSF WP 2022-18**, using the repo's
`ism.decomp_engine` + `ism.decomp_pipeline`. We build the category panels from
BEA Underlying Detail (2.4.3U real quantity, 2.4.4U price, 2.4.5U nominal),
estimate the rolling reduced-form VAR per category, sign each category-month,
and aggregate into supply/demand-driven contributions (Eq. 15) and shares
(Eq. 14). See `docs/decomp_methodology.md` for the full maths→code map.

> **Quantity data.** This model needs category quantity (2.4.3U / `U20403`).
> If your BEA cache lacks it, set `PROXY = True` below to use
> `quantity = nominal/price` (reproduces the narrative; not the exact index).

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ism.decomp_engine import DecompConfig, compute_decomp, rolling_var_residuals, classify_labels, contributions, shock_shares
from ism.decomp_pipeline import build_decomp_panels, bea_wide, load_categories, core_exclusions
from ism.datasources import BeaClient
from ism.transforms import monthly_inflation

PROXY = False   # set True if BEA table U20403 (quantity) is unavailable
SCOPE = 'headline'

def panels(scope):
    if not PROXY:
        p = build_decomp_panels(scope=scope)
        return p.log_price, p.log_quantity, p.inflation, p.weights
    bea = BeaClient(); price = bea_wide(bea,'U20404'); nom = bea_wide(bea,'U20405')
    keys=[k for k in load_categories()['key'].astype(str) if k in price and k in nom]
    if scope=='core': keys=[k for k in keys if k not in core_exclusions()]
    p=price[keys].astype(float); n=nom[keys].astype(float); q=n/p
    w=n.div(n.sum(axis=1).replace(0,np.nan),axis=0)
    return np.log(p), np.log(q), monthly_inflation(p,method='pct'), w

logp, logq, infl, w = panels(SCOPE)
print(SCOPE, logp.shape)

## Baseline decomposition (Eqs. 12–15)
J = 12 lags, W = 120-month window, binary labels.

In [ ]:
res = compute_decomp(logp, logq, infl, w, DecompConfig(var_lags=12, window=120))
c = res.contrib_yoy.dropna()
print('additivity max error:', float((res.contrib['supply']+res.contrib['demand']+res.contrib['ambiguous']-res.contrib['total']).abs().max()))
c.tail()

## Figure 3 — supply- and demand-driven contributions to y/y inflation

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
x = c.index
ax.fill_between(x, 0, c['demand'], color='#27406b', label='Demand driven')
ax.fill_between(x, c['demand'], c['demand']+c['supply'], color='#f5605a', label='Supply driven')
ax.plot(x, c['total'], color='k', lw=1, label='Total')
ax.axhline(0, color='gray', lw=.6); ax.set_ylabel('percent (y/y)'); ax.legend(ncol=3); ax.set_title(f'PCE {SCOPE}: supply vs demand contributions'); plt.show()

## Figure 2 — share of PCE by shock type (Eq. 14)

In [ ]:
sh = res.shares.dropna()
fig, ax = plt.subplots(figsize=(11,3.5))
ax.plot(sh.index, sh['supply'].rolling(5,center=True).mean(), color='#f5605a', label='supply share')
ax.plot(sh.index, sh['demand'].rolling(5,center=True).mean(), color='#27406b', label='demand share')
ax.set_ylabel('share of PCE'); ax.legend(); ax.set_title('Share of categories by shock type (5m MA)'); plt.show()
print('mean supply share:', round(float(sh['supply'].mean()),3))

## Figure 5 — precision labeling (ambiguous)
Raising the cut moves near-zero residuals into 'ambiguous'. The FRBSF published
series uses cut = 0.1.

In [ ]:
resP = compute_decomp(logp, logq, infl, w, DecompConfig(var_lags=12, window=120, precision_cut=0.1))
cp = resP.contrib_yoy.dropna(); x = cp.index
fig, ax = plt.subplots(figsize=(11,4))
ax.fill_between(x, 0, cp['demand'], color='#27406b', label='Demand')
ax.fill_between(x, cp['demand'], cp['demand']+cp['supply'], color='#f5605a', label='Supply')
ax.fill_between(x, cp['demand']+cp['supply'], cp['demand']+cp['supply']+cp['ambiguous'], color='#c9b458', label='Ambiguous')
ax.plot(x, cp['total'], color='k', lw=1, label='Total'); ax.axhline(0,color='gray',lw=.6); ax.legend(ncol=4); ax.set_title('Precision labeling (cut=0.1)'); plt.show()

## Table 1 — robustness cross-correlations
Contemporaneous correlation of the supply-driven y/y contribution across
alternative specifications (baseline, AR-3, AR-24, IRF h=2/3, first-diff).

In [ ]:
def supply_yoy(cfg, lp=logp, lq=logq):
    r = compute_decomp(lp, lq, infl, w, cfg)
    return r.contrib_yoy['supply']

specs = {
  'baseline': DecompConfig(var_lags=12, window=120),
  'AR-3':     DecompConfig(var_lags=3,  window=120),
  'AR-24':    DecompConfig(var_lags=24, window=120),
  'IRF h=2':  DecompConfig(var_lags=12, window=120, irf_h=2),
  'IRF h=3':  DecompConfig(var_lags=12, window=120, irf_h=3),
}
S = pd.DataFrame({k: supply_yoy(v) for k,v in specs.items()})
S.corr().round(3)

## Validation vs the FRBSF published series
Compares our contributions to FRBSF's chart CSVs (downloaded to
`data/raw/external/frbsf/`). The published series uses the precision (cut=0.1)
variant, so we validate `resP`.

In [ ]:
from ism.decomp_validate import validate_decomp
validate_decomp(resP.contrib_yoy, scope=SCOPE, kind='yoy')

## Section 4 — local projections (Eq. 21)
Cumulative response of the supply/demand contributions to externally identified
shocks. Drop the shock CSVs into `data/raw/external/` (see `ism.decomp_shocks`).
This cell is skipped gracefully if the shock files are absent.

In [ ]:
from ism.decomp_shocks import load_gss_monetary, load_bh_oil_supply, macro_controls, recession_peak_dummy
from ism.decomp_projection import run_section4
mp = load_gss_monetary(); oil = load_bh_oil_supply()
ctrl = None
try: ctrl = macro_controls()
except Exception: pass
rec = recession_peak_dummy(res.contrib.index)
out = run_section4(res.contrib, monetary_shock=mp, oil_shock=oil, controls=ctrl, recession_peak=rec)
if 'monetary->demand' in out:
    r = out['monetary->demand']
    plt.figure(figsize=(7,3)); plt.plot(r['h'], r['beta']); plt.fill_between(r['h'], r['lo_90'], r['hi_90'], alpha=.2)
    plt.axhline(0,color='gray',lw=.6); plt.title('Demand contribution response to HFI monetary tightening'); plt.xlabel('months'); plt.show()
else:
    print('No monetary shock file found; recession LP available for:', [k for k in out if k.startswith("recession")])